## 비교

| 데이터셋                      | 구성              |       행/열 | 장점                           | 단점                         | 해석력   | 최종 활용 적합도  |
| ------------------------- | --------------- | --------: | ---------------------------- | -------------------------- | ----- | ---------- |
| dataset1_raw_only         | 원본 거시변수만 사용     | 4799 × 14 | 변수 의미가 직관적이고 설명하기 쉬움         | 유가의 단기 움직임, 변동성, 모멘텀 반영 부족 | 높음    | 보조 비교용     |
| dataset2_raw_plus_derived | 원본 거시변수 + 파생변수  | 4799 × 28 | 원본 변수의 해석력과 파생변수의 설명력을 함께 가짐 | 변수 수가 늘어 다중공선성 가능성 있음      | 높음    | **메인 분석용** |
| dataset3_diff_only        | 1차 차분 변수 중심     | 4799 × 13 | 정상성 측면에서 안정적                 | 변수 의미가 직관적으로 약해짐           | 낮음~중간 | 보조 비교용     |
| dataset4_derived_full     | 차분 변수 + 파생변수 전체 | 4799 × 28 | 변화량 중심 분석에 적합                | 해석이 가장 복잡하고 과적합 가능성 있음     | 중간    | 보조 비교용     |


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# 1. 데이터 불러오기
# =========================

DATA_DIR = Path("../data/Finance_Final")

dataset_files = {
    "dataset1_raw_only": "dataset1_raw_only.csv",
    "dataset2_raw_plus_derived": "dataset2_raw_plus_derived.csv",
    "dataset3_diff_only": "dataset3_diff_only.csv",
    "dataset4_derived_full": "dataset4_derived_full.csv"
}

target_col = "oil_diff_target"
date_col = "date"

datasets = {}

for name, file in dataset_files.items():
    path = DATA_DIR / file
    df = pd.read_csv(path)
    
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(date_col).reset_index(drop=True)
    
    datasets[name] = df

print("데이터 불러오기 완료")
for name, df in datasets.items():
    print(name, df.shape)

데이터 불러오기 완료
dataset1_raw_only (4799, 14)
dataset2_raw_plus_derived (4799, 28)
dataset3_diff_only (4799, 13)
dataset4_derived_full (4799, 28)


In [2]:
# =========================
# 2. 데이터셋 기본 구조 비교
# =========================

summary_rows = []

for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c != target_col]
    
    summary_rows.append({
        "dataset": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "numeric_cols": len(numeric_cols),
        "feature_count": len(feature_cols),
        "missing_values": df.isna().sum().sum(),
        "start_date": df[date_col].min() if date_col in df.columns else None,
        "end_date": df[date_col].max() if date_col in df.columns else None,
        "target_mean": df[target_col].mean() if target_col in df.columns else None,
        "target_std": df[target_col].std() if target_col in df.columns else None
    })

dataset_summary = pd.DataFrame(summary_rows)

dataset_summary

,dataset,rows,cols,numeric_cols,feature_count,missing_values,start_date,end_date,target_mean,target_std
0,dataset1_raw_only,4799,14,13,12,1,2007-02-01,2026-03-16,0.007511,2.012598
1,dataset2_raw_plus_derived,4799,28,27,26,1,2007-02-01,2026-03-16,0.007511,2.012598
2,dataset3_diff_only,4799,13,12,11,1,2007-02-01,2026-03-16,0.007511,2.012598
3,dataset4_derived_full,4799,28,27,26,1,2007-02-01,2026-03-16,0.007511,2.012598


In [4]:
# =========================
# 3. 컬럼 구성 비교
# =========================

column_info = []

for name, df in datasets.items():
    cols = df.columns.tolist()
    
    column_info.append({
        "dataset": name,
        "columns": cols
    })

for info in column_info:
    print("=" * 80)
    print(info["dataset"])
    print("=" * 80)
    print(info["columns"])
    print()

dataset1_raw_only
['date', 'oil_diff_target', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TermSpread', 'TreasuryYield', 'FedFundsRate']

dataset2_raw_plus_derived
['date', 'oil_diff_target', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TermSpread', 'TreasuryYield', 'FedFundsRate', 'oil_diff_lag1', 'oil_diff_lag5', 'oil_volatility_5', 'oil_volatility_20', 'MA_5', 'MA_20', 'MA_ratio', 'MA_5_gt_MA_20', 'oil_momentum_5', 'oil_momentum_20', 'TermSpread_inversion', 'vix_high', 'is_monday', 'is_friday']

dataset3_diff_only
['date', 'oil_diff_target', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TreasuryYield', 'FedFundsRate']

dataset4_derived_full
['date', 'oil_diff_target', 'OilPrice', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX',

In [5]:
# =========================
# 4. 데이터셋별 변수 포함 여부 비교표
# =========================

all_columns = sorted(set().union(*[set(df.columns) for df in datasets.values()]))

column_compare = pd.DataFrame(index=all_columns)

for name, df in datasets.items():
    column_compare[name] = column_compare.index.isin(df.columns)

column_compare = column_compare.replace({
    True: "O",
    False: "-"
})

column_compare

,dataset1_raw_only,dataset2_raw_plus_derived,dataset3_diff_only,dataset4_derived_full
CPE,O,O,O,O
CPI,O,O,O,O
DollarIndex,O,O,O,O
FedFundsRate,O,O,O,O
IndustryProduction,O,O,O,O
MA_20,-,O,-,O
MA_5,-,O,-,O
MA_5_gt_MA_20,-,O,-,O
MA_ratio,-,O,-,O
OPECProduction,O,O,O,O


In [6]:
# =========================
# 5. 데이터셋별 결측치 비율 비교
# =========================

missing_compare = []

for name, df in datasets.items():
    missing_rate = df.isna().mean().sort_values(ascending=False)
    
    for col, rate in missing_rate.items():
        if rate > 0:
            missing_compare.append({
                "dataset": name,
                "column": col,
                "missing_rate": rate
            })

missing_compare = pd.DataFrame(missing_compare)

missing_compare

,dataset,column,missing_rate
0,dataset1_raw_only,oil_diff_target,0.000208
1,dataset2_raw_plus_derived,oil_diff_target,0.000208
2,dataset3_diff_only,oil_diff_target,0.000208
3,dataset4_derived_full,oil_diff_target,0.000208


In [7]:
# =========================
# 6. 데이터셋별 모델 성능 비교
#    시간 순서를 유지해서 train/test split
# =========================

def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))


def evaluate_dataset(df, dataset_name, target_col="oil_diff_target", date_col="date", test_size=0.2):
    df = df.copy()
    
    # 숫자형 변수만 사용
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if target_col not in numeric_cols:
        raise ValueError(f"{dataset_name}: target_col이 숫자형 컬럼에 없습니다.")
    
    feature_cols = [c for c in numeric_cols if c != target_col]
    
    # 결측치 제거
    model_df = df[[target_col] + feature_cols].dropna()
    
    X = model_df[feature_cols]
    y = model_df[target_col]
    
    split_idx = int(len(model_df) * (1 - test_size))
    
    X_train = X.iloc[:split_idx]
    X_test = X.iloc[split_idx:]
    y_train = y.iloc[:split_idx]
    y_test = y.iloc[split_idx:]
    
    models = {
        "naive_zero": None,
        "OLS": LinearRegression(),
        "Ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),
        "Lasso": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=0.001, max_iter=10000))
        ])
    }
    
    results = []
    
    for model_name, model in models.items():
        if model_name == "naive_zero":
            y_pred = np.zeros(len(y_test))
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        direction_acc = directional_accuracy(y_test, y_pred)
        
        results.append({
            "dataset": dataset_name,
            "model": model_name,
            "n_train": len(y_train),
            "n_test": len(y_test),
            "feature_count": len(feature_cols),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "Directional_Accuracy": direction_acc
        })
    
    return pd.DataFrame(results)


all_results = []

for name, df in datasets.items():
    result = evaluate_dataset(df, name, target_col=target_col, date_col=date_col)
    all_results.append(result)

model_compare = pd.concat(all_results, ignore_index=True)

model_compare

,dataset,model,n_train,n_test,feature_count,MAE,RMSE,R2,Directional_Accuracy
0,dataset1_raw_only,naive_zero,3838,960,12,1.324552,1.815916,-0.000088,0.002083
1,dataset1_raw_only,OLS,3838,960,12,1.359524,1.867847,-0.058106,0.518750
2,dataset1_raw_only,Ridge,3838,960,12,1.358316,1.866555,-0.056642,0.518750
3,dataset1_raw_only,Lasso,3838,960,12,1.337661,1.842032,-0.029060,0.516667
4,dataset2_raw_plus_derived,naive_zero,3838,960,26,1.324552,1.815916,-0.000088,0.002083
5,dataset2_raw_plus_derived,OLS,3838,960,26,1.514014,1.982499,-0.191989,0.512500
6,dataset2_raw_plus_derived,Ridge,3838,960,26,1.510771,1.978777,-0.187517,0.513542
7,dataset2_raw_plus_derived,Lasso,3838,960,26,1.491705,1.959408,-0.164383,0.516667
8,dataset3_diff_only,naive_zero,3838,960,11,1.324552,1.815916,-0.000088,0.002083
9,dataset3_diff_only,OLS,3838,960,11,1.326181,1.824917,-0.010026,0.496875


In [8]:
# =========================
# 7. 데이터셋별 최고 성능 모델 정리
# =========================

best_by_rmse = (
    model_compare
    .sort_values(["dataset", "RMSE"], ascending=[True, True])
    .groupby("dataset")
    .head(1)
    .reset_index(drop=True)
)

best_by_direction = (
    model_compare
    .sort_values(["dataset", "Directional_Accuracy"], ascending=[True, False])
    .groupby("dataset")
    .head(1)
    .reset_index(drop=True)
)

print("RMSE 기준 최고 모델")
display(best_by_rmse)

print("방향 정확도 기준 최고 모델")
display(best_by_direction)

RMSE 기준 최고 모델


,dataset,model,n_train,n_test,feature_count,MAE,RMSE,R2,Directional_Accuracy
0,dataset1_raw_only,naive_zero,3838,960,12,1.324552,1.815916,-0.000088,0.002083
1,dataset2_raw_plus_derived,naive_zero,3838,960,26,1.324552,1.815916,-0.000088,0.002083
2,dataset3_diff_only,naive_zero,3838,960,11,1.324552,1.815916,-0.000088,0.002083
3,dataset4_derived_full,naive_zero,3838,960,26,1.324552,1.815916,-0.000088,0.002083


방향 정확도 기준 최고 모델


,dataset,model,n_train,n_test,feature_count,MAE,RMSE,R2,Directional_Accuracy
0,dataset1_raw_only,OLS,3838,960,12,1.359524,1.867847,-0.058106,0.518750
1,dataset2_raw_plus_derived,Lasso,3838,960,26,1.491705,1.959408,-0.164383,0.516667
2,dataset3_diff_only,Lasso,3838,960,11,1.325866,1.824458,-0.009518,0.500000
3,dataset4_derived_full,OLS,3838,960,26,1.354217,1.874493,-0.065649,0.519792


In [9]:
# =========================
# 8. 보기 좋게 피벗 테이블 만들기
# =========================

rmse_table = model_compare.pivot(
    index="dataset",
    columns="model",
    values="RMSE"
)

direction_table = model_compare.pivot(
    index="dataset",
    columns="model",
    values="Directional_Accuracy"
)

r2_table = model_compare.pivot(
    index="dataset",
    columns="model",
    values="R2"
)

print("RMSE 비교")
display(rmse_table)

print("Directional Accuracy 비교")
display(direction_table)

print("R2 비교")
display(r2_table)

RMSE 비교


model,Lasso,OLS,Ridge,naive_zero
dataset,,,,
dataset1_raw_only,1.842032,1.867847,1.866555,1.815916
dataset2_raw_plus_derived,1.959408,1.982499,1.978777,1.815916
dataset3_diff_only,1.824458,1.824917,1.824909,1.815916
dataset4_derived_full,1.872715,1.874493,1.874467,1.815916


Directional Accuracy 비교


model,Lasso,OLS,Ridge,naive_zero
dataset,,,,
dataset1_raw_only,0.516667,0.518750,0.518750,0.002083
dataset2_raw_plus_derived,0.516667,0.512500,0.513542,0.002083
dataset3_diff_only,0.500000,0.496875,0.496875,0.002083
dataset4_derived_full,0.517708,0.519792,0.519792,0.002083


R2 비교


model,Lasso,OLS,Ridge,naive_zero
dataset,,,,
dataset1_raw_only,-0.029060,-0.058106,-0.056642,-0.000088
dataset2_raw_plus_derived,-0.164383,-0.191989,-0.187517,-0.000088
dataset3_diff_only,-0.009518,-0.010026,-0.010018,-0.000088
dataset4_derived_full,-0.063628,-0.065649,-0.065619,-0.000088


In [10]:
# =========================
# 9. 최종 비교표 저장
# =========================

OUTPUT_DIR = Path("../outputs/dataset_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dataset_summary.to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)
column_compare.to_csv(OUTPUT_DIR / "column_compare.csv")
model_compare.to_csv(OUTPUT_DIR / "model_compare.csv", index=False)
best_by_rmse.to_csv(OUTPUT_DIR / "best_by_rmse.csv", index=False)
best_by_direction.to_csv(OUTPUT_DIR / "best_by_direction.csv", index=False)

print("저장 완료:", OUTPUT_DIR)

저장 완료: ..\outputs\dataset_comparison


In [11]:
model_compare

,dataset,model,n_train,n_test,feature_count,MAE,RMSE,R2,Directional_Accuracy
0,dataset1_raw_only,naive_zero,3838,960,12,1.324552,1.815916,-0.000088,0.002083
1,dataset1_raw_only,OLS,3838,960,12,1.359524,1.867847,-0.058106,0.518750
2,dataset1_raw_only,Ridge,3838,960,12,1.358316,1.866555,-0.056642,0.518750
3,dataset1_raw_only,Lasso,3838,960,12,1.337661,1.842032,-0.029060,0.516667
4,dataset2_raw_plus_derived,naive_zero,3838,960,26,1.324552,1.815916,-0.000088,0.002083
5,dataset2_raw_plus_derived,OLS,3838,960,26,1.514014,1.982499,-0.191989,0.512500
6,dataset2_raw_plus_derived,Ridge,3838,960,26,1.510771,1.978777,-0.187517,0.513542
7,dataset2_raw_plus_derived,Lasso,3838,960,26,1.491705,1.959408,-0.164383,0.516667
8,dataset3_diff_only,naive_zero,3838,960,11,1.324552,1.815916,-0.000088,0.002083
9,dataset3_diff_only,OLS,3838,960,11,1.326181,1.824917,-0.010026,0.496875


#### model_compare
- RMSE 기준 : dataset3_diff_only : 이유 naive 제외 모델 중 RMSE가 가장 낮음
- MAE 기준 : dataset3_diff_only : MAE도 가장 낮음
- 방향 정확도 기준 : dataset4_derived_full : Directional Accuracy가 가장 높음
- 해석력 기준 : dataset1_raw_only : 원본 변수라 경제적 해석이 쉬움
- 현재 dataset2 판단 : 비추천 .. 파생변수 추가 후 성능 악화 